# PATCHR Server on Google Colab

Run the PATCHR inpainting server on Colab's free GPU and get a public URL to connect from **PATCHR-Studio**.

**Steps:**
1. Go to **Runtime > Change runtime type > GPU** (T4)
2. Run all cells below in order
3. Copy the public URL into PATCHR-Studio

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install PATCHR

In [ ]:
!git clone https://github.com/DeepFoldProtein/patchr.git
%cd patchr
!pip install -e .[cuda] -q

## 3. Install cloudflared

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!cloudflared --version

## 4. Start server and tunnel

Starts the PATCHR server, waits for it to be ready, then opens a [Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/) (no account or token required).

In [ ]:
import subprocess
import threading
import re
import time
import requests

# Start the PATCHR server
server_process = subprocess.Popen(
    ["python", "-m", "boltz.server", "--port", "31212", "--device-id", "0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait until server is ready
print("Starting PATCHR server...")
for i in range(30):
    time.sleep(2)
    try:
        r = requests.get("http://localhost:31212/api/v1/health", timeout=2)
        if r.status_code == 200:
            print(f"Server is ready! (took ~{(i+1)*2}s)")
            break
    except Exception:
        pass
else:
    print("WARNING: Server did not respond within 60s. Tunnel may show 502 errors.")

# Start cloudflared tunnel
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:31212"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

tunnel_url = None
def read_output():
    global tunnel_url
    for line in tunnel_process.stderr:
        match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line.decode("utf-8", errors="ignore"))
        if match:
            tunnel_url = match.group(1)
            break

t = threading.Thread(target=read_output, daemon=True)
t.start()
t.join(timeout=15)

if tunnel_url:
    print("\n" + "=" * 60)
    print(f"  PATCHR Server is running!")
    print(f"  Public URL: {tunnel_url}")
    print(f"\n  Copy this URL into PATCHR-Studio")
    print("=" * 60)
else:
    print("Failed to get tunnel URL.")

## 5. Health check (optional)

Verify the server is reachable through the tunnel.

In [ ]:
import requests

try:
    r = requests.get(f"{tunnel_url}/api/v1/health", timeout=10)
    print("Health check:", r.json())
except Exception as e:
    print(f"Error: {e}")

## 6. Stop server

Run this cell when you are done.

In [ ]:
tunnel_process.terminate()
server_process.terminate()
print("Server stopped.")